# 02 · Calibrazione sulle partite open di Metrica

I parametri del modello (soglie, pesi, costi di cambio stato) **non** si possono tarare sulla partita della challenge.
Li calibriamo su **altre partite**: i due *Sample Game* pubblici di Metrica Sports (stesso fornitore, tracking + eventi).

- Le etichette H/A/D di quelle partite sono ricavate **in automatico dagli eventi** (`metrica.labels_from_events`): nessuna etichettatura a mano.
- Per avvicinarle ai nostri dati togliamo la palla a blocchi lunghi (come nel tracking automatico).
- Obiettivo: lo **spellF1 ufficiale** (codice di `evaluation/possession_metrics.py`).

⚠️ Questo è il notebook pesante: ~3,6 s per configurazione per core. Eseguilo sul server.

In [ ]:
# --- Setup (locale o server Google) ---------------------------------------------
# Su Colab/Vertex: decommenta, clona la repo e indica dove hai messo i dati.
# !git clone <URL_REPO> mtah && %cd mtah
# !pip install -q -r solution/requirements.txt
# os.environ["MTAH_DATA_DIR"] = "/content/drive/MyDrive/mtah2027/data"   # PRIMA degli import
import os, sys, time
ROOT = os.environ.get("MTAH_ROOT") or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.exists(os.path.join(ROOT, "games.py")):      # notebook lanciato dalla root
    ROOT = os.getcwd()
sys.path[:0] = [ROOT, os.path.join(ROOT, "solution"), os.path.join(ROOT, "evaluation")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)
print("ROOT =", ROOT)

In [ ]:
from possession import metrica, calibrate, emissions, decode
metrica.download()
G = metrica.load_game(1)
lab = metrica.labels_from_events(G)
print(pd.Series(lab).value_counts())
s = lab[lab != "X"]; print("segmenti:", int((s[1:] != s[:-1]).sum()) + 1)

In [ ]:
# Sanity check visivo delle etichette derivate dagli eventi
codes = {"H": 0, "A": 1, "D": 2, "X": np.nan}
plt.figure(figsize=(12, 2)); plt.plot(np.arange(20000)/25, [codes[c] for c in lab[:20000]], lw=0.8)
plt.yticks([0, 1, 2], ["H", "A", "D"]); plt.xlabel("s"); plt.title("Sample Game 1, primi ~13 minuti"); plt.show()

In [ ]:
P = calibrate.prepare(seed=0)
print("parametri di default -> spellF1 (media, per partita):", calibrate.score(P, emissions.PARAMS, decode.DECODE))

## Random search (parallela)

In [ ]:
N_TRIALS = 240                 # totale
N_JOBS = os.cpu_count() or 1
SEED = 2026


t = time.time()
results = calibrate.parallel_search(N_TRIALS, N_JOBS, SEED)
print(f"{len(results)} configurazioni in {time.time()-t:.0f}s")
df = pd.DataFrame([{"spellF1": r["spell"], "g1": r["per_game"][0], "g2": r["per_game"][1], **r["decode"],
                    **{k: r["params"][k] for k in calibrate.SEARCH}} for r in results]).sort_values("spellF1", ascending=False)
df.head(10).round(3)

Scegliamo la migliore. Per evitare di premiare un colpo di fortuna, controlliamo che vada bene su **entrambe** le partite (colonne g1, g2).

In [ ]:
import json
best = max(results, key=lambda r: min(r["per_game"]))     # criterio robusto: la peggiore delle due partite
print("migliore (max-min):", round(best["spell"], 3), [round(v, 3) for v in best["per_game"]])
with open(os.path.join(ROOT, "solution", "params.json"), "w") as fh:
    json.dump({"source": "random search on Metrica Sample Games 1-2", "seed": SEED, "n_trials": len(results),
               "spell_f1_metrica": best["per_game"], "params": best["params"], "decode": best["decode"]}, fh, indent=2)
print("scritto solution/params.json")